# Secure sealed-bid auction

Pedersen commitments on secp256k1, a zero-knowledge range proof built from bit commitments and OR-proofs, additive secret sharing, and a secure comparison on shared bits (Beaver triples), so that only the winner and the winning bid are revealed.

Each cell below writes one module to disk, and the last cells run the demos. The same modules live in this repository as `.py` files together with a `tests/` folder (`pytest`), which is the version to read and run.


In [1]:
!pip install cryptography

In [2]:
%%writefile ec.py
"""
ec.py - Minimal elliptic-curve arithmetic for secp256k1 (the Bitcoin curve)

Just enough for Pedersen commitments and Sigma-protocol proofs:
point addition, negation, scalar multiplication, a compact encoding, and a
"nothing up my sleeve" hash-to-curve used to derive the second generator H.

Educational code: plain Python integers, NOT constant-time, so do not use it to
protect real secrets. tests/test_ec.py cross-checks it against the `cryptography`
library, which is what makes it trustworthy enough for this project.

Points are (x, y) tuples of ints; the point at infinity is None.
"""

import hashlib

# Curve: y^2 = x^3 + 7 over GF(P), group order N, cofactor 1
P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
G = (
    0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798,
    0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8,
)

INFINITY = None


def is_on_curve(point):
    if point is None:
        return True
    x, y = point
    return (y * y - (x * x * x + 7)) % P == 0


def point_neg(point):
    if point is None:
        return None
    x, y = point
    return (x, (-y) % P)


# --- Jacobian coordinates (X, Y, Z) represent the affine point (X/Z^2, Y/Z^3). ---
# Working in Jacobian form avoids one modular inverse per operation. Z == 0 is infinity.
_JAC_INF = (0, 1, 0)


def _jac_double(p):
    X1, Y1, Z1 = p
    if Z1 == 0 or Y1 == 0:
        return _JAC_INF
    S = 4 * X1 * Y1 * Y1 % P
    M = 3 * X1 * X1 % P
    X3 = (M * M - 2 * S) % P
    Y3 = (M * (S - X3) - 8 * pow(Y1, 4, P)) % P
    Z3 = 2 * Y1 * Z1 % P
    return (X3, Y3, Z3)


def _jac_add(p, q):
    X1, Y1, Z1 = p
    X2, Y2, Z2 = q
    if Z1 == 0:
        return q
    if Z2 == 0:
        return p
    Z1Z1 = Z1 * Z1 % P
    Z2Z2 = Z2 * Z2 % P
    U1 = X1 * Z2Z2 % P
    U2 = X2 * Z1Z1 % P
    S1 = Y1 * Z2 * Z2Z2 % P
    S2 = Y2 * Z1 * Z1Z1 % P
    if U1 == U2:
        if S1 != S2:
            return _JAC_INF          # p == -q
        return _jac_double(p)        # p == q
    H = (U2 - U1) % P
    R = (S2 - S1) % P
    H2 = H * H % P
    H3 = H * H2 % P
    V = U1 * H2 % P
    X3 = (R * R - H3 - 2 * V) % P
    Y3 = (R * (V - X3) - S1 * H3) % P
    Z3 = H * Z1 * Z2 % P
    return (X3, Y3, Z3)


def _to_jac(point):
    return _JAC_INF if point is None else (point[0], point[1], 1)


def _to_affine(p):
    X, Y, Z = p
    if Z == 0:
        return None
    zinv = pow(Z, -1, P)
    zinv2 = zinv * zinv % P
    return (X * zinv2 % P, Y * zinv2 * zinv % P)


def point_add(a, b):
    return _to_affine(_jac_add(_to_jac(a), _to_jac(b)))


def point_sub(a, b):
    return point_add(a, point_neg(b))


def scalar_mult(k, point):
    """k * point for any integer k (reduced mod the group order)."""
    k %= N
    if k == 0 or point is None:
        return None
    result = _JAC_INF
    addend = _to_jac(point)
    while k:                         # double-and-add, least significant bit first
        if k & 1:
            result = _jac_add(result, addend)
        addend = _jac_double(addend)
        k >>= 1
    return _to_affine(result)


# --- Encoding -----------------------------------------------------------------------

def encode_point(point):
    """33-byte compressed encoding (02/03 || x); a single 00 byte for infinity."""
    if point is None:
        return b"\x00"
    x, y = point
    return bytes([2 + (y & 1)]) + x.to_bytes(32, "big")


def _lift_x(x):
    """Return a point with this x (even y), or None if x is not on the curve."""
    if not 0 <= x < P:
        return None
    rhs = (pow(x, 3, P) + 7) % P
    y = pow(rhs, (P + 1) // 4, P)    # P % 4 == 3, so this is a square root if one exists
    if y * y % P != rhs:
        return None
    return (x, y if y % 2 == 0 else P - y)


def decode_point(data):
    if data == b"\x00":
        return None
    if len(data) != 33 or data[0] not in (2, 3):
        raise ValueError("bad point encoding")
    point = _lift_x(int.from_bytes(data[1:], "big"))
    if point is None:
        raise ValueError("x is not on the curve")
    if (point[1] & 1) != (data[0] & 1):
        point = (point[0], P - point[1])
    return point


def hash_to_point(seed):
    """
    Derive a curve point from a public seed by trying x = SHA256(seed || counter) until
    it lands on the curve. Nobody can know the discrete log of the result with respect
    to G, which is exactly what Pedersen commitments need from the second generator H.
    """
    counter = 0
    while True:
        digest = hashlib.sha256(seed + counter.to_bytes(4, "big")).digest()
        point = _lift_x(int.from_bytes(digest, "big") % P)
        if point is not None:
            return point
        counter += 1


Writing ec.py


In [3]:
%%writefile commitments.py
"""
commitments.py - Pedersen commitments on secp256k1

    C = bid*G + randomness*H

* Hiding:  C reveals nothing about the bid, because randomness is uniform and
           randomness*H is uniform over the group (information-theoretic).
* Binding: opening C to two different bids would reveal log_G(H). H is derived by
           hashing a public seed, so nobody knows that value, and finding it means
           solving the discrete-log problem.
* Homomorphic: C(a, r1) + C(b, r2) == C(a + b, r1 + r2). The range proof relies on this.
"""

import secrets

import ec


class CommitmentParams:
    """Public parameters: the two generators G and H and the group order."""

    SEED_H = b"secure-auction/pedersen/H/v1"

    def __init__(self):
        self.curve = "secp256k1"
        self.order = ec.N
        self.G = ec.G
        # "Nothing up my sleeve": H comes from hashing a public string, so no one
        # knows an x with H = x*G. If someone did, they could open commitments to any bid.
        self.H = ec.hash_to_point(self.SEED_H)

    def point_to_bytes(self, point):
        return ec.encode_point(point)


def generate_commitment_params():
    """Generate (deterministically derive) the public parameters for commitments."""
    return CommitmentParams()


def commit_bid(bid, randomness, params):
    """Create the Pedersen commitment C = bid*G + randomness*H (a curve point)."""
    return ec.point_add(
        ec.scalar_mult(bid, params.G),
        ec.scalar_mult(randomness, params.H),
    )


def verify_commitment(commitment, bid, randomness, params):
    """Check that a commitment opens to the claimed bid and randomness."""
    return commit_bid(bid, randomness, params) == commitment


def commitment_to_hex(commitment):
    """Short printable form of a commitment."""
    return ec.encode_point(commitment).hex()


def generate_randomness():
    """A uniformly random blinding factor in [0, group order)."""
    return secrets.randbelow(ec.N)


Writing commitments.py


In [4]:
%%writefile zkproofs.py
"""
zkproofs.py - Zero-knowledge range proof for a Pedersen commitment

Claim proved: the committed bid v satisfies  min_bid <= v <= max_bid,
without revealing v. No trusted setup, built only from the commitment scheme.

How it works
------------
1. To show x >= 0 and x < 2^n, write x in binary, x = sum(b_i * 2^i), and commit to each
   bit:  C_i = b_i*G + r_i*H, choosing the r_i so that  sum(2^i * C_i) = x*G + r*H,
   i.e. the weighted sum of the bit commitments *equals* the commitment to x.
   The verifier checks that equation. It is exactly what ties the bits to the bid.
2. For each C_i the prover shows it commits to 0 or 1, with a Fiat-Shamir "OR proof"
   (Cramer-Damgard-Schoenmakers): knowledge of r_i with C_i = r_i*H  OR  C_i - G = r_i*H.
   The branch that is not true is simulated, so the verifier cannot tell which one is real.
3. Do this twice: for x = bid - min (so bid >= min) and for x = max - bid (so bid <= max).
   Both are computed from the same commitment C, using
        C - min*G   commits to bid - min      and      max*G - C   commits to max - bid.

Everything is bound to the statement (commitment, range, bidder context) through the
Fiat-Shamir hash, so a proof cannot be replayed for another commitment or another bidder.
"""

import hashlib
import secrets

import ec

DOMAIN = b"secure-auction/range-proof/v1"


# ---------------------------------------------------------------------------------
# Fiat-Shamir challenge
# ---------------------------------------------------------------------------------
def _serialize(item):
    if isinstance(item, bytes):
        data = b"B" + item
    elif isinstance(item, str):
        data = b"S" + item.encode()
    elif isinstance(item, int):
        data = b"I" + item.to_bytes(max(1, (item.bit_length() + 7) // 8), "big")
    else:                                   # a curve point or None
        data = b"P" + ec.encode_point(item)
    return len(data).to_bytes(4, "big") + data    # length prefix: no ambiguity between fields


def _challenge(*items):
    digest = hashlib.sha256(DOMAIN + b"".join(_serialize(i) for i in items)).digest()
    return int.from_bytes(digest, "big") % ec.N


# ---------------------------------------------------------------------------------
# OR proof that a commitment holds a bit
# ---------------------------------------------------------------------------------
def _prove_bit(C, bit, r, params, binding):
    """
    Prove C = bit*G + r*H with bit in {0, 1}. The two statements are
        Y0 = C        = r*H   (bit = 0)
        Y1 = C - G    = r*H   (bit = 1)
    Returns (e0, e1, z0, z1).
    """
    G, H = params.G, params.H
    Y = [C, ec.point_sub(C, G)]
    fake = 1 - bit

    # Simulate the false branch: pick its challenge and response, solve for its commitment.
    e_fake = secrets.randbelow(ec.N)
    z_fake = secrets.randbelow(ec.N)
    A = [None, None]
    A[fake] = ec.point_sub(ec.scalar_mult(z_fake, H), ec.scalar_mult(e_fake, Y[fake]))

    # Honest branch: a normal Schnorr commitment.
    k = secrets.randbelow(ec.N)
    A[bit] = ec.scalar_mult(k, H)

    e_total = _challenge(binding, C, A[0], A[1])
    e_real = (e_total - e_fake) % ec.N          # the two challenges must add up to e_total
    z_real = (k + e_real * r) % ec.N

    e, z = [0, 0], [0, 0]
    e[bit], z[bit] = e_real, z_real
    e[fake], z[fake] = e_fake, z_fake
    return (e[0], e[1], z[0], z[1])


def _verify_bit(C, proof, params, binding):
    e0, e1, z0, z1 = proof
    if not all(isinstance(v, int) and 0 <= v < ec.N for v in (e0, e1, z0, z1)):
        return False
    G, H = params.G, params.H
    # Recompute each branch's commitment from (challenge, response), as the verifier must.
    A0 = ec.point_sub(ec.scalar_mult(z0, H), ec.scalar_mult(e0, C))
    A1 = ec.point_sub(ec.scalar_mult(z1, H), ec.scalar_mult(e1, ec.point_sub(C, G)))
    return (e0 + e1) % ec.N == _challenge(binding, C, A0, A1)


# ---------------------------------------------------------------------------------
# Proof that a committed value lies in [0, 2^n)
# ---------------------------------------------------------------------------------
def _bit_commitments(x, r_x, n, params):
    """Commit to the n bits of x so that sum(2^i * C_i) commits to x with randomness r_x."""
    bits = [(x >> i) & 1 for i in range(n)]
    rs = [secrets.randbelow(ec.N) for _ in range(n - 1)]
    partial = sum(rs[i] * (1 << i) for i in range(n - 1))
    rs.append((r_x - partial) * pow(1 << (n - 1), -1, ec.N) % ec.N)
    Cs = [ec.point_add(ec.scalar_mult(b, params.G), ec.scalar_mult(r, params.H))
          for b, r in zip(bits, rs)]
    return bits, rs, Cs


def _bit_binding(binding, index):
    return binding + _serialize(index)


def _prove_nonnegative(x, r_x, C_x, n, params, binding):
    bits, rs, Cs = _bit_commitments(x, r_x, n, params)
    proofs = [_prove_bit(C, b, r, params, _bit_binding(binding, i))
              for i, (C, b, r) in enumerate(zip(Cs, bits, rs))]
    part = {"bit_commitments": Cs, "bit_proofs": proofs}
    return part, bits, rs


def _verify_nonnegative(part, C_x, n, params, binding):
    Cs, proofs = part["bit_commitments"], part["bit_proofs"]
    if len(Cs) != n or len(proofs) != n:
        return False
    # The bits must add up to the committed value ...
    total = None
    for i, C in enumerate(Cs):
        total = ec.point_add(total, ec.scalar_mult(1 << i, C))
    if total != C_x:
        return False
    # ... and each one must really be a bit.
    return all(_verify_bit(C, p, params, _bit_binding(binding, i))
               for i, (C, p) in enumerate(zip(Cs, proofs)))


# ---------------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------------
def range_bits(min_bid, max_bid):
    """Number of bits needed for the width of the range."""
    return (max_bid - min_bid).bit_length()


def _binding(commitment, min_bid, max_bid, context, side):
    """Everything the proof is about, as bytes that get hashed into every challenge."""
    return b"".join(_serialize(item) for item in
                    (commitment, min_bid, max_bid, context, side))


def generate_range_proof_with_witness(bid, randomness, min_bid, max_bid, params, context=b""):
    """
    Returns (proof, witness). The proof is public. The witness (the bits of bid - min and
    the randomness behind their commitments) stays with the bidder, who secret-shares it
    with the servers (see auction.py).
    """
    if not 0 <= min_bid < max_bid:
        raise ValueError("need 0 <= min_bid < max_bid")
    if bid < min_bid or bid > max_bid:
        raise ValueError(f"Bid {bid} is not in range [{min_bid}, {max_bid}]")

    n = range_bits(min_bid, max_bid)
    C = ec.point_add(ec.scalar_mult(bid, params.G), ec.scalar_mult(randomness, params.H))

    C_low = ec.point_sub(C, ec.scalar_mult(min_bid, params.G))       # commits to bid - min
    C_high = ec.point_sub(ec.scalar_mult(max_bid, params.G), C)      # commits to max - bid

    low, low_bits, low_rs = _prove_nonnegative(
        bid - min_bid, randomness, C_low, n, params, _binding(C, min_bid, max_bid, context, "low"))
    high, _, _ = _prove_nonnegative(
        max_bid - bid, -randomness % ec.N, C_high, n, params,
        _binding(C, min_bid, max_bid, context, "high"))

    proof = {"min_bid": min_bid, "max_bid": max_bid, "n_bits": n, "low": low, "high": high}
    witness = {"bits": low_bits, "randomness": low_rs}
    return proof, witness


def generate_range_proof(bid, randomness, min_bid, max_bid, params, context=b""):
    """Prove min_bid <= bid <= max_bid for the commitment to `bid` with `randomness`."""
    proof, _ = generate_range_proof_with_witness(bid, randomness, min_bid, max_bid, params, context)
    return proof


def verify_range_proof(proof, commitment, params, min_bid, max_bid, context=b""):
    """
    Check that `commitment` hides a value in [min_bid, max_bid]. The range to enforce is
    passed in by the verifier: never trust the range stated inside the proof, or a bidder
    could "prove" membership in a range of their own choosing.

    Returns False for any invalid or malformed proof; it never raises on
    attacker-supplied input.
    """
    try:
        if not (0 <= min_bid < max_bid):
            return False
        n = range_bits(min_bid, max_bid)
        if (proof["min_bid"], proof["max_bid"], proof["n_bits"]) != (min_bid, max_bid, n):
            return False
        if commitment is None or not ec.is_on_curve(commitment):
            return False

        C_low = ec.point_sub(commitment, ec.scalar_mult(min_bid, params.G))
        C_high = ec.point_sub(ec.scalar_mult(max_bid, params.G), commitment)
        return (
            _verify_nonnegative(proof["low"], C_low, n, params,
                                _binding(commitment, min_bid, max_bid, context, "low"))
            and _verify_nonnegative(proof["high"], C_high, n, params,
                                    _binding(commitment, min_bid, max_bid, context, "high"))
        )
    except (KeyError, TypeError, ValueError, AttributeError, IndexError, OverflowError):
        return False


def create_and_verify_proof(bid, randomness, min_bid, max_bid, params):
    """Helper: commit, prove, verify. Returns (commitment, proof, is_valid)."""
    from commitments import commit_bid

    commitment = commit_bid(bid, randomness, params)
    try:
        proof = generate_range_proof(bid, randomness, min_bid, max_bid, params)
        return commitment, proof, verify_range_proof(proof, commitment, params, min_bid, max_bid)
    except ValueError:
        return commitment, None, False


Writing zkproofs.py


In [5]:
%%writefile secret_sharing.py
"""
secret_sharing.py - Additive secret sharing

A value v is split into shares that add up to v modulo a prime. Any proper subset of the
shares is uniformly random and says nothing about v.

The modulus is the order of the secp256k1 group (a 256-bit prime) so that shares of a
bid's bits line up with the Pedersen commitments to those bits (see auction.py).
"""

import secrets

import ec


def get_large_prime():
    """The modulus for all shares: the (prime) order of the secp256k1 group."""
    return ec.N


def share_value(value, p=None, num_shares=3):
    """Split a value into additive shares modulo p."""
    p = ec.N if p is None else p
    shares = [secrets.randbelow(p) for _ in range(num_shares - 1)]
    shares.append((value - sum(shares)) % p)
    return shares


def reconstruct(shares, p=None):
    """Recombine shares into the original value."""
    p = ec.N if p is None else p
    return sum(shares) % p


Writing secret_sharing.py


In [6]:
%%writefile mpc.py
"""
mpc.py - Secure comparison and maximum over additively shared bits

Setting: 3 servers, each holding one additive share of every value (semi-honest, all
three needed to reconstruct anything). Linear operations (add, subtract, multiply by a
public constant) are local. Multiplying two shared values uses a Beaver triple, which is
the only place servers exchange anything: they open d = x - a and e = y - b, which are
uniformly random because a and b are secret one-time masks. Nothing else is ever opened
except the final output (winner index and winning bid).

Assumption: the Beaver triples (a, b, c = a*b) come from a trusted dealer. In a real
deployment they would be produced by an offline phase (oblivious transfer or homomorphic
encryption). This code simulates the dealer.

Bids are compared bit by bit. Revealing a difference of two bids, as a plain "subtract
the shares and open" comparison does, tells everyone bid_i - bid_j and so exposes every
losing bid once the winning bid is known. Here only the final result is opened.
"""

import secrets

import ec


class MPC:
    """A simulation of 3 servers computing on additive shares modulo p."""

    def __init__(self, num_parties=3, p=None):
        self.n = num_parties
        self.p = ec.N if p is None else p
        self.opened = []            # (label, value) for every value revealed to the servers
        self.triples_used = 0

    # ---- sharing and opening -----------------------------------------------------
    def share(self, value):
        shares = [secrets.randbelow(self.p) for _ in range(self.n - 1)]
        shares.append((value - sum(shares)) % self.p)
        return shares

    def constant(self, value):
        """A sharing of a public constant (no randomness needed)."""
        return [value % self.p] + [0] * (self.n - 1)

    def open(self, shares, label):
        """All servers reveal their shares of one value. Every call is logged."""
        value = sum(shares) % self.p
        self.opened.append((label, value))
        return value

    # ---- local (free) operations ---------------------------------------------------
    def add(self, x, y):
        return [(a + b) % self.p for a, b in zip(x, y)]

    def sub(self, x, y):
        return [(a - b) % self.p for a, b in zip(x, y)]

    def mul_const(self, x, c):
        return [(a * c) % self.p for a in x]

    # ---- multiplication with a Beaver triple ----------------------------------------
    def _triple(self):
        """Trusted dealer: shares of random a, b and of c = a*b."""
        a = secrets.randbelow(self.p)
        b = secrets.randbelow(self.p)
        self.triples_used += 1
        return self.share(a), self.share(b), self.share(a * b % self.p)

    def mul(self, x, y):
        a, b, c = self._triple()
        d = self.open(self.sub(x, a), "beaver d = x - a (masked)")
        e = self.open(self.sub(y, b), "beaver e = y - b (masked)")
        # x*y = (d + a)(e + b) = d*e + d*b + e*a + c
        z = [(c[i] + d * b[i] + e * a[i]) % self.p for i in range(self.n)]
        z[0] = (z[0] + d * e) % self.p
        return z

    # ---- comparison and selection ----------------------------------------------------
    def greater_than(self, a_bits, b_bits):
        """
        Shared bit [A > B] for two numbers given as lists of shared bits, least
        significant bit first. Scanning from the lowest bit: where the bits differ the
        higher number is the one with a 1, otherwise the answer from the lower bits stays.
        """
        assert len(a_bits) == len(b_bits) and a_bits
        one = self.constant(1)
        result = self.sub(a_bits[0], self.mul(a_bits[0], b_bits[0]))      # a0 * (1 - b0)
        for a_i, b_i in zip(a_bits[1:], b_bits[1:]):
            ab = self.mul(a_i, b_i)
            differ = self.sub(self.add(a_i, b_i), self.mul_const(ab, 2))  # a XOR b
            result = self.add(result, self.mul(differ, self.sub(a_i, result)))
        return result

    def select(self, condition, if_true, if_false):
        """if_false + condition * (if_true - if_false), without learning the condition."""
        return self.add(if_false, self.mul(condition, self.sub(if_true, if_false)))

    # ---- the auction primitive --------------------------------------------------------
    def argmax_bits(self, all_bits):
        """
        all_bits[i] is bidder i's bid as a list of shared bits (least significant first).
        Returns (winner_index, winning_value). Ties go to the earlier bidder. Only the
        winner's index and bits are opened; every comparison result stays secret-shared.
        """
        if not all_bits:
            return None, None
        best_bits = all_bits[0]
        best_index = self.constant(0)
        for i in range(1, len(all_bits)):
            beats_best = self.greater_than(all_bits[i], best_bits)
            best_bits = [self.select(beats_best, new, old)
                         for new, old in zip(all_bits[i], best_bits)]
            best_index = self.select(beats_best, self.constant(i), best_index)

        winner = self.open(best_index, "output: winner index")
        value = 0
        for position, bit in enumerate(best_bits):
            value |= self.open(bit, "output: winning bid bit") << position
        return winner, value

    def summary(self):
        masked = sum(1 for label, _ in self.opened if label.startswith("beaver"))
        outputs = [(label, v) for label, v in self.opened if label.startswith("output")]
        return {"beaver_triples": self.triples_used,
                "masked_values_opened": masked,
                "output_values_opened": len(outputs)}


Writing mpc.py


In [7]:
%%writefile auction.py
"""
auction.py - Secure sealed-bid auction

Protocol
--------
1. Commit.   Each bidder publishes a Pedersen commitment C = bid*G + r*H.
2. Prove.    ... and a zero-knowledge proof that min_bid <= bid <= max_bid. The proof
             contains commitments C_i to the bits of (bid - min_bid).
3. Share.    The bidder secret-shares every bit (and the randomness of its commitment)
             among 3 servers. Each server can publish b_i*G + r_i*H for its shares; the
             sum over the servers must equal C_i. That check ties the shared bits to the
             proven commitment, so a bidder cannot commit to one bid and share another.
4. Compute.  The servers find the highest bid with a secure comparison on the shared bits
             (mpc.py). Only the winner and the winning bid are ever revealed.

Bids that fail the proof or the share check are rejected before they take part.
"""

import ec
from commitments import generate_commitment_params, commit_bid, generate_randomness, commitment_to_hex
from zkproofs import generate_range_proof_with_witness, verify_range_proof, range_bits
from secret_sharing import share_value, get_large_prime
from mpc import MPC

NUM_SERVERS = 3


def shares_match_commitments(bit_commitments, bit_shares, randomness_shares, params):
    """
    Server-side check that the shares each server received open the bit commitments from
    the proof: for every bit, the sum over servers of (b_ij*G + r_ij*H) equals C_i.
    Every server can publish its own term without revealing its share.
    """
    if len(bit_shares) != len(bit_commitments) or len(randomness_shares) != len(bit_commitments):
        return False
    for C_i, b_shares, r_shares in zip(bit_commitments, bit_shares, randomness_shares):
        if len(b_shares) != NUM_SERVERS or len(r_shares) != NUM_SERVERS:
            return False
        if not all(isinstance(v, int) and 0 <= v < ec.N for v in list(b_shares) + list(r_shares)):
            return False
        total = None
        for b_ij, r_ij in zip(b_shares, r_shares):
            total = ec.point_add(total, commit_bid(b_ij, r_ij, params))
        if total != C_i:
            return False
    return True


class SecureAuction:
    """Main auction system (bidders and servers are simulated in one process)."""

    def __init__(self, min_bid=100, max_bid=1000, verbose=True):
        self.min_bid = min_bid
        self.max_bid = max_bid
        self.verbose = verbose
        self.params = generate_commitment_params()
        self.n_bits = range_bits(min_bid, max_bid)

        # Only what the servers legitimately hold: public commitments and proofs, and
        # the shares. A bidder's plaintext bid and randomness are never stored here.
        self.bidders = {}       # bidder_id -> {'commitment', 'proof'}
        self.bit_shares = {}    # bidder_id -> [[share per server] for each bit]
        self.last_mpc = None

        self._say(f"🎯 Auction initialized")
        self._say(f"   Valid bid range: [{min_bid}, {max_bid}]  ({self.n_bits}-bit comparison)\n")

    def _say(self, text=""):
        if self.verbose:
            print(text)

    @staticmethod
    def _context(bidder_id):
        return f"auction/bidder/{bidder_id}".encode()

    # ---- bidder side ------------------------------------------------------------------
    def prepare_submission(self, bidder_id, bid):
        """
        What an honest bidder sends: commitment, range proof, and the secret shares.
        Raises ValueError if the bid is outside the allowed range (no valid proof exists).
        """
        randomness = generate_randomness()
        commitment = commit_bid(bid, randomness, self.params)
        proof, witness = generate_range_proof_with_witness(
            bid, randomness, self.min_bid, self.max_bid, self.params, self._context(bidder_id))
        return {
            "commitment": commitment,
            "proof": proof,
            "bit_shares": [share_value(b, ec.N, NUM_SERVERS) for b in witness["bits"]],
            "randomness_shares": [share_value(r, ec.N, NUM_SERVERS) for r in witness["randomness"]],
        }

    # ---- server side ------------------------------------------------------------------
    def submit(self, bidder_id, submission):
        """Verify a submission and, if it is valid, accept the bidder. Returns True/False."""
        if bidder_id in self.bidders:
            self._say("   ✗ Duplicate bidder id")
            return False
        try:
            commitment, proof = submission["commitment"], submission["proof"]
            bit_shares, randomness_shares = submission["bit_shares"], submission["randomness_shares"]
        except (KeyError, TypeError):
            self._say("   ✗ Malformed submission")
            return False

        if not verify_range_proof(proof, commitment, self.params, self.min_bid, self.max_bid,
                                  self._context(bidder_id)):
            self._say("   ✗ Range proof did not verify")
            return False
        try:
            bit_commitments = proof["low"]["bit_commitments"]
        except (KeyError, TypeError):
            return False
        if not shares_match_commitments(bit_commitments, bit_shares, randomness_shares, self.params):
            self._say("   ✗ Shares do not match the committed bits")
            return False

        self.bidders[bidder_id] = {"commitment": commitment, "proof": proof}
        self.bit_shares[bidder_id] = bit_shares
        return True

    def register_bidder(self, bidder_id, bid):
        """Convenience for demos: prepare an honest submission and submit it."""
        self._say(f"📝 Registering Bidder {bidder_id} with bid {bid}...")
        try:
            submission = self.prepare_submission(bidder_id, bid)
        except ValueError as error:
            self._say(f"   ✗ Invalid bid: {error}\n")
            return False
        self._say(f"   ✓ Commitment {commitment_to_hex(submission['commitment'])[:24]}...")
        self._say(f"   ✓ Range proof generated ({2 * self.n_bits} bit commitments)")
        accepted = self.submit(bidder_id, submission)
        if accepted:
            self._say("   ✓ Proof verified, shares match the commitment")
            self._say(f"   ✓ Bid split into {NUM_SERVERS} shares per bit\n")
        else:
            self._say("")
        return accepted

    def run_auction(self):
        """Run the secure comparison and announce the winner."""
        self._say("=" * 60)
        self._say("🚀 RUNNING SECURE AUCTION")
        self._say("=" * 60)

        if not self.bit_shares:
            self._say("No bidders registered!")
            return None, None

        bidder_ids = list(self.bit_shares)
        self._say(f"\n📊 Total bidders: {len(bidder_ids)}")
        self._say("🔒 Comparing bids bit by bit on secret shares...\n")

        mpc = MPC(NUM_SERVERS, ec.N)
        winner_index, offset = mpc.argmax_bits([self.bit_shares[b] for b in bidder_ids])
        self.last_mpc = mpc

        winner_id = bidder_ids[winner_index]
        winning_bid = offset + self.min_bid
        stats = mpc.summary()

        self._say("=" * 60)
        self._say("🎉 AUCTION RESULTS")
        self._say("=" * 60)
        self._say(f"🏆 Winner: Bidder {winner_id}")
        self._say(f"💰 Winning Bid: {winning_bid}")
        self._say(f"\n🔐 Opened during the run: {stats['output_values_opened']} output values "
                  f"(winner and winning bid) and {stats['masked_values_opened']} uniformly "
                  f"random Beaver values.")
        self._say("   No losing bid, and no difference between bids, was revealed.")
        self._say("=" * 60)
        return winner_id, winning_bid


def run_example_auction(bids, min_bid=100, max_bid=1000):
    """Run an example auction with given bids"""

    print("\n" + "=" * 60)
    print("🔐 SECURE SEALED-BID AUCTION")
    print("=" * 60 + "\n")

    auction = SecureAuction(min_bid, max_bid)

    for i, bid in enumerate(bids, start=1):
        auction.register_bidder(i, bid)

    return auction.run_auction()


if __name__ == "__main__":
    print("\n🔹 EXAMPLE 1: Bidders = [150, 920, 600]")
    run_example_auction([150, 920, 600])

    print("\n🔹 EXAMPLE 2: Bidders = [350, 350, 100, 300]")
    run_example_auction([350, 350, 100, 300])

    print("\n🔹 EXAMPLE 3: Bidders = [10, 999, 300, 700, 2000]")
    run_example_auction([10, 999, 300, 700, 2000])


Writing auction.py


In [8]:
!python auction.py


🔹 EXAMPLE 1: Bidders = [150, 920, 600]

🔐 SECURE SEALED-BID AUCTION

🎯 Auction initialized
   Valid bid range: [100, 1000]  (10-bit comparison)

📝 Registering Bidder 1 with bid 150...


   ✓ Commitment 03767c2297fd7d010d3a12b3...
   ✓ Range proof generated (20 bit commitments)


   ✓ Proof verified, shares match the commitment
   ✓ Bid split into 3 shares per bit

📝 Registering Bidder 2 with bid 920...


   ✓ Commitment 0227b78543d0cfd437da7e27...
   ✓ Range proof generated (20 bit commitments)


   ✓ Proof verified, shares match the commitment
   ✓ Bid split into 3 shares per bit

📝 Registering Bidder 3 with bid 600...


...

   ✓ Proof verified, shares match the commitment
   ✓ Bid split into 3 shares per bit

📝 Registering Bidder 3 with bid 300...


   ✓ Commitment 022bd94122cb75597ef8cdf9...
   ✓ Range proof generated (20 bit commitments)


   ✓ Proof verified, shares match the commitment
   ✓ Bid split into 3 shares per bit

📝 Registering Bidder 4 with bid 700...


   ✓ Commitment 02322f1692a2efb8d9c61c2a...
   ✓ Range proof generated (20 bit commitments)


   ✓ Proof verified, shares match the commitment
   ✓ Bid split into 3 shares per bit

📝 Registering Bidder 5 with bid 2000...
   ✗ Invalid bid: Bid 2000 is not in range [100, 1000]

🚀 RUNNING SECURE AUCTION

📊 Total bidders: 3
🔒 Comparing bids bit by bit on secret shares...

🎉 AUCTION RESULTS
🏆 Winner: Bidder 2
💰 Winning Bid: 999

🔐 Opened during the run: 11 output values (winner and winning bid) and 120 uniformly random Beaver values.
   No losing bid, and no difference between bids, was revealed.


In [9]:
%%writefile auction_enhanced.py
"""
auction_enhanced.py - Auction with step-by-step output, timings and an interactive mode

Same protocol as auction.py (real Pedersen commitments, a real range proof, secure
comparison on shared bits); this file adds measurements and a more detailed printout.
"""

import time

import ec
from auction import SecureAuction, NUM_SERVERS
from commitments import commitment_to_hex
from mpc import MPC


class EnhancedSecureAuction(SecureAuction):
    """SecureAuction plus per-bidder timings and a detailed report."""

    def __init__(self, min_bid=100, max_bid=1000, auction_name="Secure Auction"):
        print("\n" + "═" * 70)
        print(f"{'🔐 ' + auction_name.upper() + ' 🔐':^70}")
        print("═" * 70)
        super().__init__(min_bid, max_bid, verbose=False)
        self.auction_name = auction_name
        self.registration_times = []
        self.rejected_bids = 0
        print(f"🎯 {auction_name} initialized")
        print(f"   Valid bid range: [{min_bid}, {max_bid}] ({self.n_bits}-bit comparison)")
        print(f"   Cryptography: Pedersen commitments on {self.params.curve}, bit-decomposition")
        print(f"   range proofs, {NUM_SERVERS}-server MPC with Beaver triples\n")

    def register_bidder(self, bidder_id, bid, show_details=True):
        start = time.time()
        if show_details:
            print(f"\n{'─' * 70}\n📝 Registering Bidder {bidder_id}\n{'─' * 70}")

        try:
            submission = self.prepare_submission(bidder_id, bid)
        except ValueError as error:
            self.rejected_bids += 1
            if show_details:
                print(f"✗ Step 1/3: no valid range proof exists\n  └─ Reason: {error}\n  └─ Status: ❌ BID REJECTED")
            return False

        if show_details:
            print("✓ Step 1/3: commitment and range proof created")
            print(f"  └─ Commitment: {commitment_to_hex(submission['commitment'])[:40]}...")
            print(f"  └─ Proving: {self.min_bid} ≤ bid ≤ {self.max_bid} without revealing the bid")

        if not self.submit(bidder_id, submission):
            self.rejected_bids += 1
            if show_details:
                print("✗ Step 2/3: verification failed\n  └─ Status: ❌ BID REJECTED")
            return False

        elapsed = time.time() - start
        self.registration_times.append(elapsed)
        if show_details:
            print("✓ Step 2/3: range proof verified, shares match the committed bits")
            print(f"✓ Step 3/3: each bit split into {NUM_SERVERS} shares, one per server")
            print(f"⏱️  Prove + verify + share check: {elapsed * 1000:.0f}ms")
            print(f"✅ Bidder {bidder_id} registered")
        return True

    def run_auction(self):
        print("\n" + "═" * 70 + "\n🚀 AUCTION COMPUTATION PHASE\n" + "═" * 70)
        if not self.bit_shares:
            print("\n❌ No valid bidders registered!")
            return None, None

        print(f"\n📊 Statistics: {len(self.bit_shares)} bidders registered, "
              f"{self.rejected_bids} rejected")
        if self.registration_times:
            avg = sum(self.registration_times) / len(self.registration_times)
            print(f"   Average registration time: {avg * 1000:.0f}ms")

        bidder_ids = list(self.bit_shares)
        mpc = MPC(NUM_SERVERS, ec.N)
        start = time.time()
        winner_index, offset = mpc.argmax_bits([self.bit_shares[b] for b in bidder_ids])
        mpc_time = time.time() - start
        self.last_mpc = mpc

        winner_id, winning_bid = bidder_ids[winner_index], offset + self.min_bid
        stats = mpc.summary()

        print("\n" + "═" * 70 + "\n🎉 AUCTION RESULTS\n" + "═" * 70)
        print(f"  🏆 Winner:       Bidder {winner_id}")
        print(f"  💰 Winning bid:  {winning_bid}")
        print(f"  👥 Bidders:      {len(bidder_ids)}")
        print(f"\n⏱️  MPC computation: {mpc_time * 1000:.0f}ms "
              f"({stats['beaver_triples']} secure multiplications)")
        print("\n🔐 What was revealed:")
        print(f"   • the winner and the winning bid ({stats['output_values_opened']} opened values)")
        print(f"   • {stats['masked_values_opened']} values that are uniformly random one-time masks")
        print("   • NOT the losing bids, and NOT the differences between bids")
        print("═" * 70)
        return winner_id, winning_bid

    def display_summary(self):
        """Public view: commitments only. The bids themselves are not stored anywhere."""
        print("\n" + "═" * 70 + "\n📋 REGISTERED BIDDERS (public view)\n" + "═" * 70)
        print(f"\n{'Bidder ID':<12} {'Status':<14} {'Commitment':<44}")
        print("─" * 70)
        for bidder_id, info in self.bidders.items():
            print(f"{str(bidder_id):<12} {'✅ Accepted':<14} "
                  f"{commitment_to_hex(info['commitment'])[:40] + '...':<44}")
        print("═" * 70)


def interactive_mode():
    """Let the user type bids into their own auction."""
    print("\n" + "🎮" * 35 + "\nINTERACTIVE AUCTION MODE\n" + "🎮" * 35)
    try:
        min_bid = int(input("Enter minimum bid (default 100): ") or "100")
        max_bid = int(input("Enter maximum bid (default 1000): ") or "1000")
        num_bidders = int(input("How many bidders? (2-10): ") or "3")
        if not 2 <= num_bidders <= 10:
            print("⚠️  Using 3 bidders")
            num_bidders = 3
        if not 0 <= min_bid < max_bid:
            raise ValueError("need 0 <= min < max")
    except (ValueError, EOFError):
        print("⚠️  Using defaults: min=100, max=1000, bidders=3")
        min_bid, max_bid, num_bidders = 100, 1000, 3

    auction = EnhancedSecureAuction(min_bid, max_bid, "Interactive Auction")
    print(f"\n💰 Enter {num_bidders} bids:")
    for i in range(1, num_bidders + 1):
        try:
            auction.register_bidder(i, int(input(f"  Bidder {i}'s bid: ")), show_details=False)
        except (ValueError, EOFError):
            print("    ✗ Invalid input, skipping")
            continue
        print("    ✓ Registered" if i in auction.bidders else "    ✗ Rejected (outside the allowed range)")

    if auction.bit_shares:
        auction.display_summary()
        auction.run_auction()
    else:
        print("\n❌ No valid bids to auction!")


def run_example_auction_enhanced(bids, min_bid=100, max_bid=1000, name="Example Auction"):
    auction = EnhancedSecureAuction(min_bid, max_bid, name)
    print(f"\n{'═' * 70}\n📝 REGISTRATION PHASE\n{'═' * 70}")
    for i, bid in enumerate(bids, start=1):
        auction.register_bidder(i, bid, show_details=True)
    auction.display_summary()
    return auction.run_auction()


if __name__ == "__main__":
    print("\n" + "█" * 70 + "\nENHANCED SECURE SEALED-BID AUCTION SYSTEM\n" + "█" * 70)

    print("\n\n🔹 EXAMPLE 1: Standard Auction")
    run_example_auction_enhanced([150, 920, 600], name="Standard Auction")

    print("\n\n🔹 EXAMPLE 2: Tie-Breaking Auction")
    run_example_auction_enhanced([350, 350, 100, 300], name="Tie-Breaking Auction")

    print("\n\n🔹 EXAMPLE 3: Security Test (Invalid Bids)")
    run_example_auction_enhanced([10, 999, 300, 700, 2000], name="Security Test Auction")

    print("\n\n" + "█" * 70)
    try:
        answer = input("\n🎮 Want to try INTERACTIVE MODE? (yes/no): ")
    except EOFError:
        answer = "no"
    if answer.strip().lower() in ("yes", "y"):
        interactive_mode()

    print("\n" + "█" * 70 + "\nALL DEMONSTRATIONS COMPLETE ✅\n" + "█" * 70)


Writing auction_enhanced.py

In [10]:
# Answers "no" to the interactive-mode prompt so the notebook runs unattended.
# For the interactive mode, run `python auction_enhanced.py` in a terminal.
!echo no | python auction_enhanced.py


██████████████████████████████████████████████████████████████████████
ENHANCED SECURE SEALED-BID AUCTION SYSTEM
██████████████████████████████████████████████████████████████████████


🔹 EXAMPLE 1: Standard Auction

══════════════════════════════════════════════════════════════════════
                         🔐 STANDARD AUCTION 🔐                         
══════════════════════════════════════════════════════════════════════
🎯 Standard Auction initialized
   Valid bid range: [100, 1000] (10-bit comparison)
   Cryptography: Pedersen commitments on secp256k1, bit-decomposition
   range proofs, 3-server MPC with Beaver triples


══════════════════════════════════════════════════════════════════════
📝 REGISTRATION PHASE
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
📝 Registering Bidder 1
──────────────────────────────────────────────────────────────────────


✓ Step 1/3: commitment and range proof created
  └─ Commitment: 036f2fa6a076265b6f19b598b644e8d308383b36...
  └─ Proving: 100 ≤ bid ≤ 1000 without revealing the bid


✓ Step 2/3: range proof verified, shares match the committed bits
✓ Step 3/3: each bit split into 3 shares, one per server
⏱️  Prove + verify + share check: 400ms
✅ Bidder 1 registered

──────────────────────────────────────────────────────────────────────
📝 Registering Bidder 2
──────────────────────────────────────────────────────────────────────


✓ Step 1/3: commitment and range proof created
  └─ Commitment: 02d7fd0b894716fbaec91648d01afebad108f6a8...
  └─ Proving: 100 ≤ bid ≤ 1000 without revealing the bid


✓ Step 2/3: range proof verified, shares match the committed bits
✓ Step 3/3: each bit split into 3 shares, one per server
⏱️  Prove + verify + share check: 407ms
✅ Bidder 2 registered

──────────────────────────────────────────────────────────────────────
📝 Registering Bidder 3
──────────────────────────────────────────────────────────────────────


...

✓ Step 2/3: range proof verified, shares match the committed bits
✓ Step 3/3: each bit split into 3 shares, one per server
⏱️  Prove + verify + share check: 387ms
✅ Bidder 2 registered

──────────────────────────────────────────────────────────────────────
📝 Registering Bidder 3
──────────────────────────────────────────────────────────────────────


✓ Step 1/3: commitment and range proof created
  └─ Commitment: 024cc5b8ab0eb6db3837652074dbac4ff9a4d539...
  └─ Proving: 100 ≤ bid ≤ 1000 without revealing the bid


✓ Step 2/3: range proof verified, shares match the committed bits
✓ Step 3/3: each bit split into 3 shares, one per server
⏱️  Prove + verify + share check: 455ms
✅ Bidder 3 registered

──────────────────────────────────────────────────────────────────────
📝 Registering Bidder 4
──────────────────────────────────────────────────────────────────────


✓ Step 1/3: commitment and range proof created
  └─ Commitment: 033fe829dadbf14f3391bd50bc15844bcc674eb8...
  └─ Proving: 100 ≤ bid ≤ 1000 without revealing the bid


✓ Step 2/3: range proof verified, shares match the committed bits
✓ Step 3/3: each bit split into 3 shares, one per server
⏱️  Prove + verify + share check: 436ms
✅ Bidder 4 registered

──────────────────────────────────────────────────────────────────────
📝 Registering Bidder 5
──────────────────────────────────────────────────────────────────────
✗ Step 1/3: no valid range proof exists
  └─ Reason: Bid 2000 is not in range [100, 1000]
  └─ Status: ❌ BID REJECTED

══════════════════════════════════════════════════════════════════════
📋 REGISTERED BIDDERS (public view)
══════════════════════════════════════════════════════════════════════

Bidder ID    Status         Commitment                                  
──────────────────────────────────────────────────────────────────────
2            ✅ Accepted     036cbbe068636183e89c6615218dc13de508c108... 
3            ✅ Accepted     024cc5b8ab0eb6db3837652074dbac4ff9a4d539... 
4            ✅ Accepted     033fe829dadbf14f3391bd50bc15844bcc